# Лекция: Бинарная (логистическая) регрессия в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 9** (адаптация с языка R на Python)

## Краткая теория

**Логистическая регрессия** — когда отклик принимает **только два значения** (0/1, болен/здоров, принять/отказать).

Модель (logit):

P(Y=1 | x) = 1 / (1 + exp(-(b0 + b1*x1 + ... + bk*xk)))

H0: между переменными нет логистической зависимости.

В Python: `statsmodels.formula.api.logit` / `glm(..., family=Binomial())`  
или `sklearn.linear_model.LogisticRegression`.


## 0. Импорт библиотек


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Пример: приём в университет (UCLA binary)

- **admit** — принят (1) / нет (0)
- **gre** — балл экзамена
- **gpa** — средний балл
- **rank** — престиж школы (1 — высший, 4 — низший)

В R: `read.csv("http://www.ats.ucla.edu/stat/data/binary.csv")`


In [ ]:
url = "https://stats.idre.ucla.edu/stat/data/binary.csv"
mydata = pd.read_csv(url)
print(mydata.head())
print("\nРазмерность:", mydata.shape)
print("\nadmit:")
print(mydata["admit"].value_counts())


In [ ]:
mydata["rank"] = mydata["rank"].astype("category")
print(mydata.dtypes)


### Логит-модель

В R: `glm(admit ~ gre + gpa + rank, family = binomial("logit"))`


In [ ]:
mylogit = smf.logit("admit ~ gre + gpa + C(rank)", data=mydata).fit()
print(mylogit.summary())


### Пробит-модель

В R: `family = binomial("probit")`


In [ ]:
myprobit = smf.probit("admit ~ gre + gpa + C(rank)", data=mydata).fit()
print(myprobit.summary())


### Значимость модели в целом (LR-тест)


In [ ]:
ll_null = mylogit.llnull
ll_model = mylogit.llf
lr_stat = 2 * (ll_model - ll_null)
df = mylogit.df_model
p_lr = stats.chi2.sf(lr_stat, df)
print(f"LR chi2 = {lr_stat:.2f}, df = {df:.0f}, p = {p_lr:.4e}")
print("Модель значима в целом" if p_lr < 0.05 else "Модель НЕ значима")
print("\nКоэффициенты (logit):")
print(mylogit.params)


### Матрица неточностей (confusion matrix)

В R: `confusionMatrix` из **caret**. Порог: 0.5


In [ ]:
prob = mylogit.predict(mydata)
pred = (prob >= 0.5).astype(int)

cm = confusion_matrix(mydata["admit"], pred)
print("Confusion matrix:")
print(cm)

print("\nAccuracy =", accuracy_score(mydata["admit"], pred).round(4))
print(classification_report(mydata["admit"], pred, digits=3))

ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap="Blues")
plt.title("Матрица неточностей (logit, threshold=0.5)")
plt.show()


**Расшифровка матрицы:**
- **TN** — верно предсказан 0
- **FP** — ошибка I рода
- **FN** — ошибка II рода
- **TP** — верно предсказан 1

**Accuracy** = (TP + TN) / N  
**Sensitivity (Recall)** = TP / (TP + FN)  
**Specificity** = TN / (TN + FP)


### ROC-кривая и AUC

В R: пакет **pROC**


In [ ]:
fpr, tpr, thresholds = roc_curve(mydata["admit"], prob)
auc = roc_auc_score(mydata["admit"], prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Случайный классификатор")
plt.xlabel("1 - Specificity (FPR)")
plt.ylabel("Sensitivity (TPR)")
plt.title("ROC-кривая (logit)")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC = {auc:.4f}")
print("Чем ближе AUC к 1, тем лучше модель; 0.5 — как подбрасывание монеты.")


---
## 2. Задание: train / test + логит-модель

В задании используется таблица **datlg** (кардиология: y — делать операцию или нет).  
Если файла нет — применяем тот же пайплайн к UCLA binary (или подставьте свой CSV).


In [ ]:
# === Подставьте datlg при наличии ===
# datlg = pd.read_csv("datlg.csv")

data = mydata.copy()
y = data["admit"]
X = data[["gre", "gpa", "rank"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

train = X_train.copy()
train["admit"] = y_train.values
test = X_test.copy()
test["admit"] = y_test.values

print("Train:", train.shape, "Test:", test.shape)
print("Доля admit=1 (train):", y_train.mean().round(3))
print("Доля admit=1 (test):", y_test.mean().round(3))


In [ ]:
model = smf.logit("admit ~ gre + gpa + C(rank)", data=train).fit(disp=False)
print(model.summary())


In [ ]:
def evaluate(model, df, y_true, name):
    prob = model.predict(df)
    pred = (prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, pred)
    auc = roc_auc_score(y_true, prob)
    cm = confusion_matrix(y_true, pred)
    print(f"=== {name} ===")
    print("Confusion matrix:\n", cm)
    print(f"Accuracy = {acc:.4f}, AUC = {auc:.4f}")
    print(classification_report(y_true, pred, digits=3))
    return prob, pred, auc

prob_tr, pred_tr, auc_tr = evaluate(model, train, y_train, "TRAIN")
prob_te, pred_te, auc_te = evaluate(model, test, y_test, "TEST")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, prob, title, auc_val in [
    (axes[0], y_train, prob_tr, "Train", auc_tr),
    (axes[1], y_test, prob_te, "Test", auc_te),
]:
    fpr, tpr, _ = roc_curve(y_true, prob)
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc_val:.3f}")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(f"ROC ({title})")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Как описать результаты (шаблон)

1. **Коэффициенты:** какие предикторы значимы (p < 0.05), знак связи.
2. **Качество на train:** accuracy, sensitivity, specificity, AUC.
3. **Качество на test:** те же метрики — нет ли переобучения.
4. **ROC:** насколько кривая выше диагонали, величина AUC.
5. **Вывод:** пригодна ли модель (AUC > 0.7 — приемлемо, > 0.8 — хорошо).

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `glm(..., family=binomial("logit"))` | `smf.logit("y ~ x", data=df).fit()` |
| `glm(..., family=binomial("probit"))` | `smf.probit("y ~ x", data=df).fit()` |
| `summary(glm)` | `model.summary()` |
| `fitted.values` | `model.predict(df)` |
| `ifelse(p < 0.5, 0, 1)` | `(prob >= 0.5).astype(int)` |
| `confusionMatrix` (caret) | `sklearn.metrics.confusion_matrix` |
| `roc` / `auc` (pROC) | `roc_curve`, `roc_auc_score` |
| train/test split | `train_test_split(..., stratify=y)` |

---
## Рекомендации

1. Для **datlg**: `df = pd.read_csv("datlg.csv")`, укажите столбец отклика и предикторы.
2. Используйте `stratify=y` при разбиении.
3. Порог 0.5 можно менять (смотрите ROC).
4. Установка: `pip install statsmodels scikit-learn`

**Удачи с выполнением Задания 9!**
